# Credit Risk Feature Engineering Pipeline

**Dataset source:**  
Manu Siddhartha, November 6, 2020, *Bondora Peer-to-Peer Lending Data*, IEEE Dataport.  
doi: https://dx.doi.org/10.21227/33kz-0s65

---

## Notebook Overview

This notebook walks through an end-to-end machine learning pipeline for **credit risk assessment**.  
The goal is to predict the **probability of default** for a loan applicant using a LightGBM model.

**Tasks covered:**
1. Load and split data
2. Explore and impute numerical variables
3. Engineer income and debt features
4. Engineer datetime features
5. Evaluate categorical cardinality
6. Impute categorical variables
7. Group rare categories
8. Encode categorical variables
9. Train and evaluate LightGBM
10. Evaluate feature importance
11. Build end-to-end pipeline
12. Score latest customer cohort

## Task 1 — Load and Split the Data

Load the historical loan application data and split it 80/20 into training and test sets.  
The target variable `default` indicates whether a customer has defaulted (1) or not (0).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

# Reproducibility seed
seed = 10

In [ ]:
# Load data
df = pd.read_csv("loan_data.csv", low_memory=False)
df.head()

In [ ]:
# Split into 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    df.drop("default", axis=1),
    df["default"],
    test_size=0.20,
    random_state=seed,
)

X_train.shape, X_test.shape

In [ ]:
# Calculate the default rate in each dataset
# ~65% of customers defaulted — this is our class imbalance
y_train.mean(), y_test.mean()

## Task 2 — Evaluate and Impute Numerical Variables

Identify numerical variables (continuous and discrete), examine their distributions,
find any with missing data, and impute them using -1 as a sentinel value.

We use -1 because LightGBM can learn that -1 means "missing" — it is outside the
natural range of all these variables.

In [ ]:
from feature_engine.imputation import ArbitraryNumberImputer

In [ ]:
# Find all numerical variables
var_num = list(X_train.select_dtypes(include="number").columns)
var_num

In [ ]:
# Find discrete variables (fewer than 20 unique values)
var_discrete = [var for var in var_num if X_train[var].nunique() < 20]
var_discrete

In [ ]:
# Examine discrete variable value frequencies
for var in var_discrete:
    print(var)
    print(X_train[var].value_counts(normalize=True))
    print()

In [ ]:
# Examine numerical variable distributions
fig = X_train[var_num].hist(bins=50, figsize=(15, 15))
plt.tight_layout()
plt.show()

In [ ]:
# Identify numerical variables with missing data
var_num_na = [var for var in var_num if X_train[var].isnull().sum() > 0]
var_num_na

In [ ]:
# Inspect range of missing variables
# All values are >= 0, so -1 is a safe sentinel that won't clash with real data
X_train[var_num_na].describe()

In [ ]:
# Set up the imputer — replace missing numerical values with -1
imputer = ArbitraryNumberImputer(
    arbitrary_number=-1,
    variables=var_num_na,
)

# Fit on train, transform both sets (prevents data leakage)
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Confirm no missing values remain
X_test[var_num_na].isnull().sum()

## Task 3 — Create Income and Debt Variables

Create three new features that capture the relationship between income and debt:
- **total_income**: sum of all income sources
- **dti** (debt-to-income ratio): proportion of income used to service debt
- **cash** (discretionary income): money left after paying monthly debt

> Note: customers with zero income did not disclose it. Division by zero produces `inf` and `nan`,
> which we replace with 0 using `np.where`.

In [ ]:
# Income source columns
var_income = [
    "income_from_employer",
    "income_from_pension",
    "income_from_family_allowance",
    "income_from_social_welfare",
    "income_from_leave_pay",
    "income_from_child_support",
    "income_other",
]

In [ ]:
# Total income = sum of all income sources
X_train["total_income"] = X_train[var_income].sum(axis=1)
X_test["total_income"] = X_test[var_income].sum(axis=1)

X_train[var_income + ["total_income"]].head()

In [ ]:
# Plot income distribution
X_train["total_income"].hist(bins=50, figsize=(8, 5))
plt.ylabel("Number of people")
plt.xlabel("Total income")
plt.title("Total income distribution")
plt.show()

In [ ]:
# How many customers have zero income (did not disclose)?
print(f"Total customers: {X_train.shape[0]}")
print(f"Zero income customers: {X_train[X_train['total_income'] == 0].shape[0]}")

In [ ]:
# Default rate is similar regardless of income disclosure
print(f"Default rate (zero income): {y_train[X_train['total_income'] == 0].mean():.3f}")
print(f"Default rate (income declared): {y_train[X_train['total_income'] != 0].mean():.3f}")

In [ ]:
# Debt-to-income ratio: what fraction of income goes to debt payments?
# Division by zero (zero income) produces inf/nan — fixed with np.where
X_train["dti"] = X_train["total_debt"].div(X_train["total_income"])
X_test["dti"] = X_test["total_debt"].div(X_test["total_income"])

# FIX: replace inf and nan with 0 (no income = no ratio)
X_train["dti"] = np.where(np.isfinite(X_train["dti"]), X_train["dti"], 0)
X_test["dti"] = np.where(np.isfinite(X_test["dti"]), X_test["dti"], 0)

X_train[["dti", "total_debt", "total_income"]].head()

In [ ]:
# Discretionary income: money left after paying monthly debt
X_train["cash"] = X_train["total_income"].sub(X_train["total_debt"])
X_test["cash"] = X_test["total_income"].sub(X_test["total_debt"])

X_train[["cash", "total_debt", "total_income"]].head()

## Task 4 — Create Features from Datetime Variables

We have two datetime fields:
- `application_date`: when the customer applied (date + time)
- `date_of_birth`: used to compute age at application

The company only lends to customers aged 18+. We remove any under-18 customers found in the data.  
We then extract calendar features: hour, day of week, day of month, and month.

In [ ]:
# Datetime columns
vars_temp = ["application_date", "date_of_birth"]

# Check for missing values — none expected
X_train[vars_temp].isnull().sum()

In [ ]:
# Compute age at time of application (in whole years)
X_train["age"] = (
    (
        pd.to_datetime(X_train["application_date"])
        - pd.to_datetime(X_train["date_of_birth"])
    ).dt.days / 365
).astype(int)

X_test["age"] = (
    (
        pd.to_datetime(X_test["application_date"])
        - pd.to_datetime(X_test["date_of_birth"])
    ).dt.days / 365
).astype(int)

X_train["age"].describe()

In [ ]:
# Age distribution
X_train["age"].hist(bins=50)
plt.xlabel("Age")
plt.ylabel("Number of people")
plt.title("Age distribution at application")
plt.show()

In [ ]:
# Check for under-18 customers (should not receive loans)
print(f"Under-18 in train: {(X_train['age'] < 18).sum()}")
print(f"Under-18 in test: {(X_test['age'] < 18).sum()}")
X_train[X_train["age"] < 18][["age", "application_date", "date_of_birth"]].head()

In [ ]:
# Remove under-18 customers from both datasets and realign target
X_train = X_train[X_train["age"] >= 18]
X_test = X_test[X_test["age"] >= 18]

y_train = y_train.loc[X_train.index]
y_test = y_test.loc[X_test.index]

X_train.shape, X_test.shape

In [ ]:
# Extract calendar features from the application date
for dataset in [X_train, X_test]:
    dataset["application_date"] = pd.to_datetime(dataset["application_date"])
    dataset["dow"] = dataset["application_date"].dt.day_of_week   # 0=Monday
    dataset["dom"] = dataset["application_date"].dt.day            # day of month
    dataset["month"] = dataset["application_date"].dt.month
    dataset["hour"] = dataset["application_date"].dt.hour

X_train[["application_date", "dow", "dom", "month", "hour"]].head()

In [ ]:
# Drop the raw datetime columns — we have extracted everything useful
X_train.drop(["application_date", "date_of_birth"], axis=1, inplace=True)
X_test.drop(["application_date", "date_of_birth"], axis=1, inplace=True)

In [ ]:
# Visualise application patterns across time features
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))
plt.subplots_adjust(hspace=0.5)
fig.suptitle("Number of applications by time feature", fontsize=16, y=0.95)

for var, ax in zip(["hour", "dow", "dom", "month"], axs.ravel()):
    X_train.groupby(var)[var].count().plot(ax=ax)
    ax.set_title(var)
    ax.set_ylabel("Applications")

plt.show()

## Task 5 — Evaluate Categorical Variable Cardinality

Categorical variables with very many unique values (high cardinality) are problematic:
- They can introduce unseen values at prediction time
- Variables like `employment_position` have entries in multiple languages

We identify and remove these high-cardinality variables.

In [ ]:
# Find categorical (string/object) variables
# Exclude datetime columns already dropped
vars_cat = list(X_train.select_dtypes(include="O").columns)
vars_cat

In [ ]:
# Visualise cardinality of each categorical variable
X_train[vars_cat].nunique().sort_values(ascending=True).plot.bar(figsize=(10, 5))
plt.ylabel("Number of unique values")
plt.title("Cardinality of categorical variables")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect the high-cardinality culprits
highly_cardinal_vars = ["employment_position", "city", "county"]

for var in highly_cardinal_vars:
    print(f"{var}: {X_train[var].nunique()} unique values")
    print(X_train[var].unique()[:20])
    print()

In [ ]:
# Drop high-cardinality variables from both datasets
X_train.drop(highly_cardinal_vars, axis=1, inplace=True)
X_test.drop(highly_cardinal_vars, axis=1, inplace=True)

# Update categorical variable list
vars_cat = [var for var in vars_cat if var not in highly_cardinal_vars]
vars_cat

## Task 6 — Impute Categorical Variables

Categorical variables with missing values are filled with the string `"missing"`.  
This preserves the information that a customer did not provide this field —  
which itself may be predictive of default.

In [ ]:
# Find categorical variables with missing data
var_cat_na = [var for var in vars_cat if X_train[var].isnull().sum() > 0]
var_cat_na

In [ ]:
# Build imputation dictionary: each missing variable → "missing"
imputation_cat_dict = {var: "missing" for var in var_cat_na}
imputation_cat_dict

In [ ]:
# FIX: use assignment instead of inplace=True to avoid pandas FutureWarning
X_train = X_train.fillna(value=imputation_cat_dict)
X_test = X_test.fillna(value=imputation_cat_dict)

# Confirm no missing values remain
X_test[var_cat_na].isnull().sum()

## Task 7 — Group Rare Categories

Categories that appear in less than 5% of observations are grouped into a single `"Rare"` label.  
This prevents the model from over-fitting to very rare categories, and handles future
unseen categories gracefully.

In [ ]:
# Plot category frequency for each categorical variable
fig, axs = plt.subplots(nrows=5, ncols=3, figsize=(15, 25))
plt.subplots_adjust(hspace=1)
fig.suptitle("Category frequency", fontsize=18, y=0.92)

for var, ax in zip(vars_cat, axs.ravel()):
    X_train[var].value_counts(normalize=True, dropna=False).plot.bar(ax=ax)
    ax.axhline(y=0.05, color="r", linestyle="-")  # 5% threshold line
    ax.set_title(var)
    ax.set_ylabel("% observations")

plt.show()

In [ ]:
# Variables where we will group rare categories (< 5% frequency)
vars_group = [
    "language",
    "use_of_loan",
    "occupation",
    "home_ownership",
    "credit_score_1",
    "credit_score_2",
    "credit_score_3",
]

In [ ]:
# Identify frequent categories (>5%) for each variable — fit on train only
frequent_cat_dict = {}

for var in vars_group:
    categories = X_train[var].value_counts(normalize=True)
    frequent_cat_dict[var] = list(categories[categories > 0.05].index)

frequent_cat_dict

In [ ]:
# Replace rare categories with "Rare" in both datasets
from numpy import where

for var in vars_group:
    X_train[var] = where(
        X_train[var].isin(frequent_cat_dict[var]), X_train[var], "Rare"
    )
    X_test[var] = where(
        X_test[var].isin(frequent_cat_dict[var]), X_test[var], "Rare"
    )

# Verify
X_test["language"].value_counts(normalize=True)

## Task 8 — Encode Categorical Variables

LightGBM's Python API does not accept string values — all features must be numeric.  
We use `OrdinalEncoder` which assigns an integer to each category.  
The encoder is fitted on training data only, then applied to both sets.

In [ ]:
import joblib
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# Fit encoder on training data
enc = OrdinalEncoder()
enc.fit(X_train[vars_cat])

In [ ]:
# Encode categorical variables in both datasets
X_train[vars_cat] = enc.transform(X_train[vars_cat])
X_test[vars_cat] = enc.transform(X_test[vars_cat])

X_train[vars_cat].head()

In [ ]:
# Confirm no missing values were introduced by encoding
X_test[vars_cat].isnull().sum()

In [ ]:
# Save encoder for use in the pipeline (Task 11)
joblib.dump(enc, "encoder.pkl")

## Task 9 — Train and Evaluate a LightGBM

Train a LightGBM classifier using the Python API.  
- Early stopping halts training if performance does not improve after 3 iterations  
- We evaluate using ROC-AUC (probability ranking quality) and a full classification report

In [ ]:
import lightgbm as lgb

In [ ]:
# Configure LightGBM model
gbm = lgb.LGBMClassifier(
    num_iterations=1000,
    random_state=seed,
)

In [ ]:
# Train with early stopping — stops if test loss doesn't improve after 3 rounds
evals_result = {}

gbm.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=vars_cat,
    callbacks=[
        lgb.early_stopping(3),
        lgb.record_evaluation(evals_result),
    ],
)

In [ ]:
# Plot learning curve — shows when training converged
ax = lgb.plot_metric(evals_result, metric="binary_logloss")
plt.title("LightGBM learning curve")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

# ROC-AUC: measures how well the model ranks defaults above non-defaults
pred_train = gbm.predict_proba(X_train)[:, 1]
pred_test = gbm.predict_proba(X_test)[:, 1]

roc_train = roc_auc_score(y_train, pred_train)
roc_test = roc_auc_score(y_test, pred_test)

print(f"Train set ROC-AUC: {roc_train:.4f}")
print(f"Test set  ROC-AUC: {roc_test:.4f}")

In [ ]:
# Classification report: precision, recall, f1-score, accuracy
pred_train_labels = gbm.predict(X_train)
pred_test_labels = gbm.predict(X_test)

print("Train set:")
print(classification_report(y_train, pred_train_labels))

print("Test set:")
print(classification_report(y_test, pred_test_labels))

In [ ]:
# Save model for use in the pipeline (Task 11)
joblib.dump(gbm, "lightGBM.pkl")

## Task 10 — Evaluate Feature Importance

We evaluate feature importance using three complementary methods:
1. **LightGBM built-in**: based on how often a feature is used to split trees
2. **Permutation importance**: measures the drop in performance when a feature is shuffled
3. **Single-feature classifiers**: trains a decision tree per feature and checks if it beats random (ROC-AUC > 0.5)

In [ ]:
# --- Method 1: LightGBM built-in importance ---
print(f"Total input features: {len(X_train.columns)}")

ax = lgb.plot_importance(gbm, figsize=(15, 10))
plt.title("LightGBM feature importance (split count)")
plt.show()

In [ ]:
# --- Method 2: Permutation importance ---
# Shuffles each feature individually and measures the drop in ROC-AUC
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    gbm,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=3,
    random_state=seed,
)

In [ ]:
# Plot permutation importance — features causing a drop in AUC when shuffled are important
perm_series = pd.Series(perm_result.importances_mean, index=X_train.columns)

perm_series.sort_values(ascending=False).plot.bar(figsize=(20, 6))
plt.ylabel("Mean drop in ROC-AUC")
plt.title("Permutation importance")
plt.tight_layout()
plt.show()

In [ ]:
# FIX: correctly filter features where permutation caused a performance drop (> 0)
# Original code: list((result > 0).index) returns ALL features regardless of value
# Correct code: filter first, then get index
feature_subset_1 = list(perm_series[perm_series > 0].index)

print(f"Features with positive permutation importance: {len(feature_subset_1)}")
print(feature_subset_1)

In [ ]:
# --- Method 3: Single-feature classifiers ---
# Train a decision tree per feature and measure its ROC-AUC via cross-validation
# Features with ROC-AUC > 0.5 are better than random guessing
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate

clf = DecisionTreeClassifier(random_state=seed)

# FIX: use a different variable name (single_feat_result) to avoid
# overwriting the permutation importance result
single_feat_result = {}

for var in X_train.columns:
    tmp = cross_validate(
        clf,
        X_train[var].to_frame(),
        y_train,
        cv=3,
        scoring="roc_auc",
    )
    single_feat_result[var] = tmp["test_score"].mean()

single_feat_result

In [ ]:
# Plot ROC-AUC per single-feature classifier
single_feat_series = pd.Series(single_feat_result, index=X_train.columns)

single_feat_series.sort_values(ascending=False).plot.bar(figsize=(20, 6))
plt.axhline(y=0.5, color="r", linestyle="--", label="Random baseline")
plt.ylabel("ROC-AUC")
plt.title("Single-feature classifier ROC-AUC")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Features that individually beat random guessing (ROC-AUC > 0.5)
# FIX: correct filtering — was same bug as feature_subset_1
feature_subset_2 = list(single_feat_series[single_feat_series > 0.5].index)

print(f"Features beating random baseline: {len(feature_subset_2)}")
print(feature_subset_2)

In [ ]:
# Final model input feature list — needed for the pipeline
print("Model input features:")
print(X_train.columns.tolist())

## Task 11 — End-to-End Feature Engineering Pipeline

Bundle all transformation and feature creation steps into a single reusable function.  
This pipeline takes **raw data** (as it arrives from the platform) and outputs  
a fully preprocessed feature matrix ready for LightGBM predictions.

---

In [ ]:
import joblib
import numpy as np
import pandas as pd

from numpy import where
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────

# All raw columns needed to create input features
FEATURES_ALL = [
    'new_customer', 'application_date', 'income_verification', 'language',
    'date_of_birth', 'gender', 'country', 'loan_amount', 'county', 'city',
    'use_of_loan', 'education', 'marital_status', 'nr_dependants',
    'employment_status', 'employment_duration', 'employment_position',
    'work_experience', 'occupation', 'home_ownership',
    'income_from_employer', 'income_from_pension',
    'income_from_family_allowance', 'income_from_social_welfare',
    'income_from_leave_pay', 'income_from_child_support', 'income_other',
    'nr_debt_items', 'total_debt', 'credit_score_1', 'credit_score_2',
    'credit_score_3', 'credit_score_4', 'nr_previous_loans',
    'amount_previous_loans', 'previous_repayments',
    'previous_early_repayments', 'previous_early_repayments_count'
]

# Final features fed to the model (order must match training)
FEATURES_INPUT = [
    'new_customer', 'income_verification', 'language', 'gender', 'country',
    'loan_amount', 'use_of_loan', 'education', 'marital_status',
    'nr_dependants', 'employment_status', 'employment_duration',
    'work_experience', 'occupation', 'home_ownership',
    'income_from_employer', 'income_from_pension',
    'income_from_family_allowance', 'income_from_social_welfare',
    'income_from_leave_pay', 'income_from_child_support', 'income_other',
    'nr_debt_items', 'total_debt', 'credit_score_1', 'credit_score_2',
    'credit_score_3', 'credit_score_4', 'nr_previous_loans',
    'amount_previous_loans', 'previous_repayments',
    'previous_early_repayments', 'previous_early_repayments_count',
    'total_income', 'dti', 'cash', 'age', 'dow', 'dom', 'month', 'hour',
]

# Income source columns
FEATURES_INCOME = [
    'income_from_employer', 'income_from_pension',
    'income_from_family_allowance', 'income_from_social_welfare',
    'income_from_leave_pay', 'income_from_child_support', 'income_other',
]

# Categorical columns to encode
FEATURES_ENCODE = [
    'income_verification', 'language', 'gender', 'country', 'use_of_loan',
    'education', 'marital_status', 'employment_status', 'employment_duration',
    'work_experience', 'occupation', 'home_ownership',
    'credit_score_1', 'credit_score_2', 'credit_score_3',
]

In [ ]:
# ── Learned parameters ────────────────────────────────────────────────────────

# Imputation values (learned from training data)
IMPUTATION_DICT = {
    # Numerical: -1 sentinel (outside natural value range)
    'nr_dependants': -1,
    'credit_score_4': -1,
    'previous_repayments': -1,
    'previous_early_repayments': -1,
    # Categorical: explicit "missing" label
    'income_verification': 'missing',   # FIX: was missing from original dict
    'gender': 'missing',
    'education': 'missing',
    'marital_status': 'missing',
    'employment_status': 'missing',
    'employment_duration': 'missing',
    'work_experience': 'missing',
    'occupation': 'missing',
    'home_ownership': 'missing',
    'credit_score_1': 'missing',
    'credit_score_2': 'missing',
    'credit_score_3': 'missing',
}

# Frequent categories per variable (categories seen in >= 5% of train data)
FREQUENT_CAT_DICT = {
    'language': ['estonian', 'finnish', 'spanish', 'russian'],
    'use_of_loan': ['unknown', 'other', 'home_improvement', 'loan_consolidation'],
    'occupation': ['missing', 'other', 'retail'],
    'home_ownership': [
        'owner', 'tenant_furnished', 'living_with_parents',
        'mortgage', 'tenant_unfurnished'
    ],
    'credit_score_1': ['missing', 'M', 'M1'],
    'credit_score_2': ['missing', 'B'],
    'credit_score_3': ['missing', 'RL2']
}

In [ ]:
def feature_engineering_pipe(df):
    """
    End-to-end feature engineering pipeline.

    Takes raw loan application data and returns a fully preprocessed
    feature matrix ready for LightGBM inference.

    Parameters
    ----------
    df : pd.DataFrame
        Raw data containing all columns listed in FEATURES_ALL.

    Returns
    -------
    pd.DataFrame
        Preprocessed features in the order expected by the model.
    """
    # Work on a copy to avoid modifying the original dataframe
    df = df[FEATURES_ALL].copy()

    # Step 1: Impute missing values
    # FIX: use assignment instead of inplace=True to avoid pandas FutureWarning
    df = df.fillna(IMPUTATION_DICT)

    # Step 2: Create income-related features
    df['total_income'] = df[FEATURES_INCOME].sum(axis=1)
    df['dti'] = df['total_debt'].div(df['total_income'])
    # FIX: replace inf/nan from division by zero using np.where
    df['dti'] = np.where(np.isfinite(df['dti']), df['dti'], 0)
    df['cash'] = df['total_income'].sub(df['total_debt'])

    # Step 3: Create datetime features
    df['age'] = (
        (
            pd.to_datetime(df['application_date'])
            - pd.to_datetime(df['date_of_birth'])
        ).dt.days / 365
    ).astype(int)

    df['application_date'] = pd.to_datetime(df['application_date'])
    df['dow'] = df['application_date'].dt.day_of_week
    df['dom'] = df['application_date'].dt.day
    df['month'] = df['application_date'].dt.month
    df['hour'] = df['application_date'].dt.hour

    # Step 4: Group infrequent categories into "Rare"
    for var in FREQUENT_CAT_DICT.keys():
        df[var] = where(
            df[var].isin(FREQUENT_CAT_DICT[var]), df[var], 'Rare'
        )

    # Step 5: Encode categorical variables as integers
    df[FEATURES_ENCODE] = enc.transform(df[FEATURES_ENCODE])

    # Return only model input features in the correct order
    return df[FEATURES_INPUT]

In [ ]:
# Load saved encoder and model
enc = joblib.load("encoder.pkl")
gbm = joblib.load("lightGBM.pkl")

In [ ]:
# Load and split data
df = pd.read_csv("loan_data.csv", low_memory=False)
seed = 10

X_train, X_test, y_train, y_test = train_test_split(
    df.drop("default", axis=1),
    df["default"],
    test_size=0.20,
    random_state=seed,
)

X_train.shape, X_test.shape

In [ ]:
# Remove under-18 customers using the indexes identified during development
train_index = [
    30394, 8998, 2634, 20910, 2560, 2434, 1148, 2448, 889,
    11656, 18764, 1166, 11647, 2565, 18823, 2475, 2403, 8802,
    9081, 2479, 11557, 19035, 1132, 2439, 30369, 2423, 1174,
    20932, 20892, 919, 18765, 2589, 8770, 11386, 20941, 20895,
    956, 11555, 930, 20925, 2509,
]

test_index = [
    915, 30378, 18914, 922, 18842, 8939, 2486, 30412, 11676, 1137,
    2440, 2447
]

X_train = X_train.drop(index=train_index)
X_test = X_test.drop(index=test_index)
y_train = y_train.drop(index=train_index)
y_test = y_test.drop(index=test_index)

X_train.shape, X_test.shape

In [ ]:
# Run raw data through the feature engineering pipeline
X_train_t = feature_engineering_pipe(X_train)
X_test_t = feature_engineering_pipe(X_test)

X_train_t.shape, X_test_t.shape

In [ ]:
# Evaluate pipeline output with the trained model
pred_train = gbm.predict_proba(X_train_t)[:, 1]
pred_test = gbm.predict_proba(X_test_t)[:, 1]

print(f"Train set ROC-AUC: {roc_auc_score(y_train, pred_train):.4f}")
print(f"Test set  ROC-AUC: {roc_auc_score(y_test, pred_test):.4f}")

In [ ]:
print("Train set:")
print(classification_report(y_train, gbm.predict(X_train_t)))

print("Test set:")
print(classification_report(y_test, gbm.predict(X_test_t)))

## Task 12 — Evaluate a Cohort of Recent Customers

Score the most recent customer applications through the pipeline.  

> **Note:** These customers just received their loans, so we do not yet have enough  
> repayment history to accurately label them. Many customers are currently labelled  
> as "no default" even though they may default in coming months.  
> Lower performance metrics here are expected and not a cause for concern.

In [ ]:
# Load latest customer data
df_latest = pd.read_csv("latest_customers.csv", low_memory=False)
df_latest.head()

In [ ]:
# Store target separately before passing through pipeline
df_target = df_latest["default"]

In [ ]:
# Transform raw data through the pipeline
df_t = feature_engineering_pipe(df_latest)
df_t.shape

In [ ]:
# ROC-AUC on latest cohort
pred_df = gbm.predict_proba(df_t)[:, 1]
roc_df = roc_auc_score(df_target, pred_df)
print(f"Latest cohort ROC-AUC: {roc_df:.4f}")

In [ ]:
# Full classification report for latest cohort
pred_df_labels = gbm.predict(df_t)
print("Latest cohort:")
print(classification_report(df_target, pred_df_labels))